# Lezione 3 — Il tuo primo classificatore di sentimenti

**Obiettivi di oggi**
- Trasformare il testo in numeri con un bag of words.
- Addestrare un classificatore che distingua le recensioni positive da quelle negative.
- Verificare l'effetto di avere più dati di addestramento.

Consulta il [capitolo 2 del manuale](../manual/02_sentiment_analysis.md) per la spiegazione completa.

## 1. Carica il corpus di recensioni cinematografiche

Usiamo `pos.txt` e `neg.txt`, inclusi nel materiale del corso: più di cinquemila frasi positive e cinquemila negative.

In [ ]:
import tensorflow as tf

with open("../pos.txt", encoding="latin-1") as f:
    positivas = [linea.strip() for linea in f if linea.strip()]
with open("../neg.txt", encoding="latin-1") as f:
    negativas = [linea.strip() for linea in f if linea.strip()]

textos = positivas + negativas
etiquetas = [1] * len(positivas) + [0] * len(negativas)

print("Frases positivas:", len(positivas))
print("Frases negativas:", len(negativas))
print("Ejemplo positivo:", positivas[0])
print("Ejemplo negativo:", negativas[0])

## 2. Dividi in addestramento e test, e mescola i dati

In [ ]:
import numpy as np

indices = np.arange(len(textos))
np.random.seed(42)
np.random.shuffle(indices)

textos = np.array(textos)[indices]
etiquetas = np.array(etiquetas)[indices]

corte = int(len(textos) * 0.8)
textos_entreno, textos_test = textos[:corte], textos[corte:]
etiquetas_entreno, etiquetas_test = etiquetas[:corte], etiquetas[corte:]

print("Frases de entrenamiento:", len(textos_entreno))
print("Frases de test:", len(textos_test))

## 3. Costruisci il vocabolario con `TextVectorization`

Questo strato conta le parole più frequenti nei tuoi dati e le converte in un bag of words.

In [ ]:
vectorizador = tf.keras.layers.TextVectorization(max_tokens=5000, output_mode="multi_hot")
vectorizador.adapt(textos_entreno)

print("Tamaño del vocabulario:", len(vectorizador.get_vocabulary()))
print("Primeras 10 palabras:", vectorizador.get_vocabulary()[:10])

## 4. Definisci e addestra il classificatore

In [ ]:
modelo = tf.keras.Sequential([
    vectorizador,
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(textos_entreno, etiquetas_entreno, epochs=10, validation_split=0.1)

## 5. Valuta e prova con frasi nuove

In [ ]:
perdida_test, exactitud_test = modelo.evaluate(textos_test, etiquetas_test)
print(f"Exactitud sobre el conjunto de test: {exactitud_test:.2%}")

frases_nuevas = [
    "una actuación brillante y una historia conmovedora",
    "el peor guion que he visto en años",
]
print(modelo.predict(frases_nuevas))

## Riepilogo

- Hai convertito il testo in vettori numerici con un bag of words, senza scrivere il vocabolario manualmente.
- Hai addestrato un classificatore binario (positivo/negativo) con un solo neurone di uscita e attivazione sigmoide.
- Il materiale originale di questa lezione mostra come la stessa pipeline, applicata a Sentiment140 (un milione seicento mila tweet), migliora la precisione grazie all'avere molti più dati.

Continua con il [notebook di pratica di questa lezione](../notebooks_practica/clase03_practica_sentimiento.ipynb).